<a href="https://colab.research.google.com/github/Jethendiran/privacy-anonymization-reidentification/blob/main/Anonymization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cs-training.csv to cs-training (2).csv


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("cs-training.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 150000
Columns: 12


In [ ]:
print(df.columns.tolist())

['Unnamed: 0', 'SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [ ]:
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


The dataset must be anonymized before being sent to the data scientists because it contains personal and sensitive financial information. Although direct identifiers such as names and addresses are not present, combinations of attributes such as age, income, debt ratio, credit utilization and payment history may distinguish an individual from other individuals.

If a combination of these attributes is unique, a person with additional external information could potentially reidentify the corresponding individual and obtain sensitive financial information.

Anonymization reduces the risk of reidentification and information leakage while preserving useful information for the data science team. The objective is to protect individuals' privacy while retaining sufficient information to build an accurate prediction model.



In [ ]:
# Q2 - Descriptive statistics

stats = pd.DataFrame(index=df.columns)

stats["Number of data points"] = df.count()
stats["Missing values"] = df.isna().sum()
stats["Minimum"] = df.min()
stats["Maximum"] = df.max()
stats["Mean"] = df.mean()
stats["Standard deviation"] = df.std()

stats = stats.round(3)

stats

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation
Unnamed: 0,150000,0,1.0,150000.0,75000.500,43301.415
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755
age,150000,0,0.0,109.0,52.295,14.772
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130


In [ ]:
# Q3 - Privacy classification of each feature

classification = pd.DataFrame({
    "Personal": [
        "N",  # Unnamed: 0
        "N",  # SeriousDlqin2yrs
        "Y",  # RevolvingUtilizationOfUnsecuredLines
        "Y",  # age
        "Y",  # NumberOfTime30-59DaysPastDueNotWorse
        "Y",  # DebtRatio
        "Y",  # MonthlyIncome
        "Y",  # NumberOfOpenCreditLinesAndLoans
        "Y",  # NumberOfTimes90DaysLate
        "Y",  # NumberRealEstateLoansOrLines
        "Y",  # NumberOfTime60-89DaysPastDueNotWorse
        "Y"   # NumberOfDependents
    ],

    "Sensitive": [
        "N",  # Unnamed: 0
        "Y",  # SeriousDlqin2yrs
        "Y",  # RevolvingUtilizationOfUnsecuredLines
        "N",  # age
        "Y",  # NumberOfTime30-59DaysPastDueNotWorse
        "Y",  # DebtRatio
        "Y",  # MonthlyIncome
        "Y",  # NumberOfOpenCreditLinesAndLoans
        "Y",  # NumberOfTimes90DaysLate
        "Y",  # NumberRealEstateLoansOrLines
        "Y",  # NumberOfTime60-89DaysPastDueNotWorse
        "N"   # NumberOfDependents
    ],

    "Type": [
        "Identifier",
        "Sensitive target",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier"
    ]
}, index=df.columns)

classification

,Personal,Sensitive,Type
Unnamed: 0,N,N,Identifier
SeriousDlqin2yrs,N,Y,Sensitive target
RevolvingUtilizationOfUnsecuredLines,Y,Y,Quasi-identifier
age,Y,N,Quasi-identifier
NumberOfTime30-59DaysPastDueNotWorse,Y,Y,Quasi-identifier
DebtRatio,Y,Y,Quasi-identifier
MonthlyIncome,Y,Y,Quasi-identifier
NumberOfOpenCreditLinesAndLoans,Y,Y,Quasi-identifier
NumberOfTimes90DaysLate,Y,Y,Quasi-identifier
NumberRealEstateLoansOrLines,Y,Y,Quasi-identifier


In [ ]:
# Combine Q2 descriptive statistics with Q3 classification

final_table = pd.concat([stats, classification], axis=1)

final_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type
Unnamed: 0,150000,0,1.0,150000.0,75000.500,43301.415,N,N,Identifier
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier


In [ ]:
# Q4 - Identification power

risk_scores = [
    0,  # Unnamed: 0
    2,  # SeriousDlqin2yrs
    2,  # RevolvingUtilizationOfUnsecuredLines
    1,  # age
    2,  # NumberOfTime30-59DaysPastDueNotWorse
    2,  # DebtRatio
    2,  # MonthlyIncome
    2,  # NumberOfOpenCreditLinesAndLoans
    2,  # NumberOfTimes90DaysLate
    2,  # NumberRealEstateLoansOrLines
    2,  # NumberOfTime60-89DaysPastDueNotWorse
    1   # NumberOfDependents
]

final_table["Identification Power"] = risk_scores

final_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type,Identification Power
Unnamed: 0,150000,0,1.0,150000.0,75000.500,43301.415,N,N,Identifier,0
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target,2
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier,2
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier,1
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier,2
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier,2
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier,2
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier,2
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier,2
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier,2


In [ ]:
# Q5 - Feature selection

keep_features = [
    "N",  # Unnamed: 0
    "Y",  # SeriousDlqin2yrs
    "Y",  # RevolvingUtilizationOfUnsecuredLines
    "Y",  # age
    "Y",  # NumberOfTime30-59DaysPastDueNotWorse
    "Y",  # DebtRatio
    "Y",  # MonthlyIncome
    "Y",  # NumberOfOpenCreditLinesAndLoans
    "Y",  # NumberOfTimes90DaysLate
    "Y",  # NumberRealEstateLoansOrLines
    "Y",  # NumberOfTime60-89DaysPastDueNotWorse
    "Y"   # NumberOfDependents
]

final_table["Keep for model?"] = keep_features

final_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type,Identification Power,Keep for model?
Unnamed: 0,150000,0,1.0,150000.0,75000.500,43301.415,N,N,Identifier,0,N
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target,2,Y
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier,2,Y
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier,1,Y
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier,2,Y
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier,2,Y
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier,2,Y
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier,2,Y
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier,2,Y
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier,2,Y


In [ ]:
# Remove the useless technical index column
df = df.drop(columns=["Unnamed: 0"])

print("Remaining columns:")
print(df.columns.tolist())

Remaining columns:
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [ ]:
# Q2 - Descriptive statistics after removing the technical index

stats = pd.DataFrame(index=df.columns)

stats["Number of data points"] = df.count()
stats["Missing values"] = df.isna().sum()
stats["Minimum"] = df.min()
stats["Maximum"] = df.max()
stats["Mean"] = df.mean()
stats["Standard deviation"] = df.std()

stats = stats.round(3)

stats

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755
age,150000,0,0.0,109.0,52.295,14.772
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130
NumberOfTime60-89DaysPastDueNotWorse,150000,0,0.0,98.0,0.240,4.155


In [ ]:
# Q3 - Privacy classification

classification = pd.DataFrame({
    "Personal": [
        "N",  # SeriousDlqin2yrs
        "Y",  # RevolvingUtilizationOfUnsecuredLines
        "Y",  # age
        "Y",  # NumberOfTime30-59DaysPastDueNotWorse
        "Y",  # DebtRatio
        "Y",  # MonthlyIncome
        "Y",  # NumberOfOpenCreditLinesAndLoans
        "Y",  # NumberOfTimes90DaysLate
        "Y",  # NumberRealEstateLoansOrLines
        "Y",  # NumberOfTime60-89DaysPastDueNotWorse
        "Y"   # NumberOfDependents
    ],

    "Sensitive": [
        "Y",  # SeriousDlqin2yrs
        "Y",  # RevolvingUtilizationOfUnsecuredLines
        "N",  # age
        "Y",  # NumberOfTime30-59DaysPastDueNotWorse
        "Y",  # DebtRatio
        "Y",  # MonthlyIncome
        "Y",  # NumberOfOpenCreditLinesAndLoans
        "Y",  # NumberOfTimes90DaysLate
        "Y",  # NumberRealEstateLoansOrLines
        "Y",  # NumberOfTime60-89DaysPastDueNotWorse
        "N"   # NumberOfDependents
    ],

    "Type": [
        "Sensitive target",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier",
        "Quasi-identifier"
    ]
}, index=df.columns)

classification

,Personal,Sensitive,Type
SeriousDlqin2yrs,N,Y,Sensitive target
RevolvingUtilizationOfUnsecuredLines,Y,Y,Quasi-identifier
age,Y,N,Quasi-identifier
NumberOfTime30-59DaysPastDueNotWorse,Y,Y,Quasi-identifier
DebtRatio,Y,Y,Quasi-identifier
MonthlyIncome,Y,Y,Quasi-identifier
NumberOfOpenCreditLinesAndLoans,Y,Y,Quasi-identifier
NumberOfTimes90DaysLate,Y,Y,Quasi-identifier
NumberRealEstateLoansOrLines,Y,Y,Quasi-identifier
NumberOfTime60-89DaysPastDueNotWorse,Y,Y,Quasi-identifier


In [ ]:
# Q4 - Identification power

risk_scores = [
    2,  # SeriousDlqin2yrs
    2,  # RevolvingUtilizationOfUnsecuredLines
    1,  # age
    2,  # NumberOfTime30-59DaysPastDueNotWorse
    2,  # DebtRatio
    2,  # MonthlyIncome
    2,  # NumberOfOpenCreditLinesAndLoans
    2,  # NumberOfTimes90DaysLate
    2,  # NumberRealEstateLoansOrLines
    2,  # NumberOfTime60-89DaysPastDueNotWorse
    1   # NumberOfDependents
]

final_table = pd.concat([stats, classification], axis=1)

final_table["Identification Power"] = risk_scores

final_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type,Identification Power
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target,2
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier,2
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier,1
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier,2
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier,2
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier,2
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier,2
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier,2
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier,2
NumberOfTime60-89DaysPastDueNotWorse,150000,0,0.0,98.0,0.240,4.155,Y,Y,Quasi-identifier,2


In [ ]:
# Q5 - Variables selected for the prediction model

keep_for_model = [
    "Y",  # SeriousDlqin2yrs - target
    "Y",  # RevolvingUtilizationOfUnsecuredLines
    "Y",  # age
    "Y",  # NumberOfTime30-59DaysPastDueNotWorse
    "Y",  # DebtRatio
    "Y",  # MonthlyIncome
    "Y",  # NumberOfOpenCreditLinesAndLoans
    "Y",  # NumberOfTimes90DaysLate
    "Y",  # NumberRealEstateLoansOrLines
    "Y",  # NumberOfTime60-89DaysPastDueNotWorse
    "Y"   # NumberOfDependents
]

final_table["Keep for model?"] = keep_for_model

final_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type,Identification Power,Keep for model?
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target,2,Y
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier,2,Y
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier,1,Y
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier,2,Y
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier,2,Y
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier,2,Y
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier,2,Y
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier,2,Y
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier,2,Y
NumberOfTime60-89DaysPastDueNotWorse,150000,0,0.0,98.0,0.240,4.155,Y,Y,Quasi-identifier,2,Y


In [ ]:
# Q6 - Anonymization / Generalization plan

anonymization_plan = pd.DataFrame({
    "Method": [
        "Keep as target",
        "Equal-frequency generalization",
        "Age-range generalization",
        "Equal-frequency generalization",
        "Equal-frequency generalization",
        "Income-range generalization",
        "Equal-frequency generalization",
        "Equal-frequency generalization",
        "Equal-frequency generalization",
        "Equal-frequency generalization",
        "Category generalization"
    ],

    "Number of groups": [
        2,
        5,
        7,
        5,
        5,
        6,
        5,
        5,
        5,
        5,
        4
    ]
}, index=df.columns)

anonymization_plan

,Method,Number of groups
SeriousDlqin2yrs,Keep as target,2
RevolvingUtilizationOfUnsecuredLines,Equal-frequency generalization,5
age,Age-range generalization,7
NumberOfTime30-59DaysPastDueNotWorse,Equal-frequency generalization,5
DebtRatio,Equal-frequency generalization,5
MonthlyIncome,Income-range generalization,6
NumberOfOpenCreditLinesAndLoans,Equal-frequency generalization,5
NumberOfTimes90DaysLate,Equal-frequency generalization,5
NumberRealEstateLoansOrLines,Equal-frequency generalization,5
NumberOfTime60-89DaysPastDueNotWorse,Equal-frequency generalization,5


In [ ]:
final_report_table = pd.concat(
    [final_table, anonymization_plan],
    axis=1
)

final_report_table

,Number of data points,Missing values,Minimum,Maximum,Mean,Standard deviation,Personal,Sensitive,Type,Identification Power,Keep for model?,Method,Number of groups
SeriousDlqin2yrs,150000,0,0.0,1.0,0.067,0.250,N,Y,Sensitive target,2,Y,Keep as target,2
RevolvingUtilizationOfUnsecuredLines,150000,0,0.0,50708.0,6.048,249.755,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
age,150000,0,0.0,109.0,52.295,14.772,Y,N,Quasi-identifier,1,Y,Age-range generalization,7
NumberOfTime30-59DaysPastDueNotWorse,150000,0,0.0,98.0,0.421,4.193,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
DebtRatio,150000,0,0.0,329664.0,353.005,2037.819,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
MonthlyIncome,120269,29731,0.0,3008750.0,6670.221,14384.674,Y,Y,Quasi-identifier,2,Y,Income-range generalization,6
NumberOfOpenCreditLinesAndLoans,150000,0,0.0,58.0,8.453,5.146,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
NumberOfTimes90DaysLate,150000,0,0.0,98.0,0.266,4.169,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
NumberRealEstateLoansOrLines,150000,0,0.0,54.0,1.018,1.130,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5
NumberOfTime60-89DaysPastDueNotWorse,150000,0,0.0,98.0,0.240,4.155,Y,Y,Quasi-identifier,2,Y,Equal-frequency generalization,5


In [20]:
def anonymize_age(x):
    if pd.isna(x):
        return "Missing"
    elif x < 25:
        return "<25"
    elif x < 35:
        return "25-34"
    elif x < 45:
        return "35-44"
    elif x < 55:
        return "45-54"
    elif x < 65:
        return "55-64"
    elif x < 75:
        return "65-74"
    else:
        return "75+"

In [21]:
def anonymize_income(x):
    if pd.isna(x):
        return "Missing"
    elif x < 2000:
        return "<2000"
    elif x < 4000:
        return "2000-3999"
    elif x < 6000:
        return "4000-5999"
    elif x < 10000:
        return "6000-9999"
    elif x < 20000:
        return "10000-19999"
    else:
        return "20000+"

In [22]:
def anonymize_dependents(x):
    if pd.isna(x):
        return "Missing"
    elif x == 0:
        return "0"
    elif x == 1:
        return "1"
    elif x == 2:
        return "2"
    else:
        return "3+"

In [23]:
def generalize_quantiles(series, groups=5):
    return pd.qcut(
        series,
        q=groups,
        duplicates="drop"
    ).astype(str)

## Q7 — Implementation of anonymization

We create three versions of the dataset with different levels of
anonymization.

- **Case A:** All predictor variables are anonymized.
- **Case B:** Variables with identification power 0 and 1 remain unchanged;
  variables with power 2 are anonymized.
- **Case C:** Variables with identification power 0, 1 and 2 remain unchanged.

The target variable `SeriousDlqin2yrs` is kept unchanged in all cases because
it is the response variable of the prediction model.

In [24]:
# Predictor variables selected for the model

predictor_columns = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

print("Number of predictor variables:", len(predictor_columns))
print(predictor_columns)

Number of predictor variables: 10
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [25]:
def generalize_quantiles(series, groups=5):
    return pd.qcut(
        series,
        q=groups,
        duplicates="drop"
    )


def anonymize_column(data, column):

    if column == "age":
        return pd.cut(
            data[column],
            bins=[0, 18, 30, 40, 50, 60, 70, 120],
            labels=[
                "0-18",
                "19-30",
                "31-40",
                "41-50",
                "51-60",
                "61-70",
                "71+"
            ],
            include_lowest=True
        )

    elif column == "MonthlyIncome":
        return pd.cut(
            data[column],
            bins=[-1, 2000, 4000, 6000, 10000, 20000, float("inf")],
            labels=[
                "0-2000",
                "2001-4000",
                "4001-6000",
                "6001-10000",
                "10001-20000",
                "20000+"
            ]
        )

    elif column == "NumberOfDependents":
        return pd.cut(
            data[column],
            bins=[-1, 0, 1, 3, float("inf")],
            labels=[
                "0",
                "1",
                "2-3",
                "4+"
            ]
        )

    else:
        return generalize_quantiles(data[column], groups=5)

### Case A — All predictor variables anonymized

In Case A, every predictor variable selected for the model is generalized.
This provides the highest level of anonymization among the three cases.

The target variable `SeriousDlqin2yrs` is not generalized.

In [26]:
# CASE A
# Anonymize all predictor variables

case_A = df.copy()

for col in predictor_columns:
    case_A[col] = anonymize_column(case_A, col)

print("Case A created successfully.")
print("Shape:", case_A.shape)

Case A created successfully.
Shape: (150000, 11)


In [27]:
print("Original age:")
print(df["age"].head(10).tolist())

print("\nCase A age:")
print(case_A["age"].head(10).tolist())

Original age:
[45, 40, 38, 30, 49, 74, 57, 39, 27, 57]

Case A age:
['41-50', '31-40', '31-40', '19-30', '41-50', '71+', '51-60', '31-40', '19-30', '51-60']


### Case B — Keep identification power 0 and 1 unchanged

In Case B, variables with identification power 0 and 1 are kept in their
original form. Variables with identification power 2 are generalized.

In our dataset, `age` and `NumberOfDependents` have identification power 1,
so they remain unchanged. The remaining predictor variables with power 2
are anonymized.

In [28]:
# CASE B
# Keep power 1 variables unchanged
# Anonymize power 2 variables

case_B = df.copy()

case_B_anonymize = [
    "RevolvingUtilizationOfUnsecuredLines",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

for col in case_B_anonymize:
    case_B[col] = anonymize_column(case_B, col)

print("Case B created successfully.")
print("Shape:", case_B.shape)

Case B created successfully.
Shape: (150000, 11)


In [29]:
print("Original age:")
print(df["age"].head(10).tolist())

print("\nCase B age:")
print(case_B["age"].head(10).tolist())

print("\nOriginal MonthlyIncome:")
print(df["MonthlyIncome"].head(10).tolist())

print("\nCase B MonthlyIncome:")
print(case_B["MonthlyIncome"].head(10).tolist())

Original age:
[45, 40, 38, 30, 49, 74, 57, 39, 27, 57]

Case B age:
[45, 40, 38, 30, 49, 74, 57, 39, 27, 57]

Original MonthlyIncome:
[9120.0, 2600.0, 3042.0, 3300.0, 63588.0, 3500.0, nan, 3500.0, nan, 23684.0]

Case B MonthlyIncome:
['6001-10000', '2001-4000', '2001-4000', '2001-4000', '20000+', '2001-4000', nan, '2001-4000', nan, '20000+']


### Case C — Keep identification power 0, 1 and 2 unchanged

In Case C, variables with identification power 0, 1 and 2 are kept in their
original form.

In our selected predictor variables, the identification powers are only 1
and 2. Therefore, no predictor variable is generalized in Case C.

The target variable `SeriousDlqin2yrs` is also kept unchanged.

In [30]:
# CASE C
# Keep power 0, 1 and 2 unchanged

case_C = df.copy()

print("Case C created successfully.")
print("Shape:", case_C.shape)

Case C created successfully.
Shape: (150000, 11)


In [31]:
print("Original age:")
print(df["age"].head(10).tolist())

print("\nCase C age:")
print(case_C["age"].head(10).tolist())

print("\nOriginal MonthlyIncome:")
print(df["MonthlyIncome"].head(10).tolist())

print("\nCase C MonthlyIncome:")
print(case_C["MonthlyIncome"].head(10).tolist())

Original age:
[45, 40, 38, 30, 49, 74, 57, 39, 27, 57]

Case C age:
[45, 40, 38, 30, 49, 74, 57, 39, 27, 57]

Original MonthlyIncome:
[9120.0, 2600.0, 3042.0, 3300.0, 63588.0, 3500.0, nan, 3500.0, nan, 23684.0]

Case C MonthlyIncome:
[9120.0, 2600.0, 3042.0, 3300.0, 63588.0, 3500.0, nan, 3500.0, nan, 23684.0]


### Saving the anonymized datasets

The three versions are saved as CSV files so that they can be used for
the reidentification analysis in Question 8.

In [32]:
# Save the three datasets

case_A.to_csv("anonymized_case_A.csv", index=False)
case_B.to_csv("anonymized_case_B.csv", index=False)
case_C.to_csv("anonymized_case_C.csv", index=False)

print("Three anonymized datasets saved successfully.")

Three anonymized datasets saved successfully.


## Q8 — Reidentification Rate

The reidentification rate is calculated for each of the three anonymization
cases.

For each individual, an identifier is created by combining the values of
the selected predictor variables.

If an identifier occurs only once in the dataset, the corresponding
individual is uniquely identifiable from the selected features.

Let:

- **A** = number of identifiers whose population is equal to 1.
- **N** = total number of records.

The reidentification rate is calculated as:

**Reidentification Rate = (A / N) × 100**

In [33]:
# Q8 - Variables used to construct the identifier

identifier_columns = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

print("Number of identifier variables:", len(identifier_columns))

Number of identifier variables: 10


In [41]:
def calculate_reidentification_rate(data, identifier_columns):

    # Create a copy containing only the identifier variables
    identifier_data = data[identifier_columns].copy()

    # Convert categorical columns to object dtype before filling missing values
    # to avoid TypeError when adding 'MISSING' as a new category.
    for col in identifier_data.select_dtypes(include='category').columns:
        identifier_data[col] = identifier_data[col].astype(object)

    # Replace missing values with a common label
    identifier_data = identifier_data.fillna("MISSING")

    # Convert each row into one combined identifier
    identifiers = identifier_data.astype(str).agg("|".join, axis=1)

    # Count how many times each identifier appears
    identifier_counts = identifiers.value_counts()

    # A = number of identifiers appearing exactly once
    A = (identifier_counts == 1).sum()

    # Total population
    N = len(data)

    # Reidentification rate
    rate = (A / N) * 100

    return A, N, rate

In [42]:
A_A, N_A, rate_A = calculate_reidentification_rate(
    case_A,
    identifier_columns
)

print("CASE A")
print("A =", A_A)
print("N =", N_A)
print("Reidentification rate =", round(rate_A, 4), "%")

/tmp/ipykernel_851/1956748377.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  identifier_data = identifier_data.fillna("MISSING")


CASE A
A = 4852
N = 150000
Reidentification rate = 3.2347 %


In [43]:
A_B, N_B, rate_B = calculate_reidentification_rate(
    case_B,
    identifier_columns
)

print("CASE B")
print("A =", A_B)
print("N =", N_B)
print("Reidentification rate =", round(rate_B, 4), "%")

/tmp/ipykernel_851/1956748377.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  identifier_data = identifier_data.fillna("MISSING")


CASE B
A = 43496
N = 150000
Reidentification rate = 28.9973 %


In [44]:
A_C, N_C, rate_C = calculate_reidentification_rate(
    case_C,
    identifier_columns
)

print("CASE C")
print("A =", A_C)
print("N =", N_C)
print("Reidentification rate =", round(rate_C, 4), "%")

CASE C
A = 149000
N = 150000
Reidentification rate = 99.3333 %


### Interpretation of the results

The results show a clear relationship between the level of generalization
and the reidentification rate.

In Case A, where all predictor variables are generalized, 4,852 records
have a unique combination of feature values. The resulting
reidentification rate is 3.2347%.

In Case B, the power-1 variables are retained in their original form while
the power-2 variables are generalized. The number of uniquely identifiable
records increases to 43,496, corresponding to a reidentification rate of
28.9973%.

In Case C, all predictor variables are kept unchanged. This produces
149,000 unique identifiers out of 150,000 records, resulting in a
reidentification rate of 99.3333%.

These results demonstrate that retaining more precise feature values
substantially increases the uniqueness of records and therefore the
potential for reidentification. Generalization reduces the number of
unique combinations by making multiple records share the same generalized
feature values.


In [45]:
case_A.to_csv("anonymized_case_A.csv", index=False)
case_B.to_csv("anonymized_case_B.csv", index=False)
case_C.to_csv("anonymized_case_C.csv", index=False)

print("All three datasets saved successfully.")

All three datasets saved successfully.


In [46]:
q8_results = pd.DataFrame({
    "Case": [
        "Case A - All anonymized",
        "Case B - Keep power 0-1",
        "Case C - Keep power 0-2"
    ],
    "A - Unique identifiers": [
        A_A,
        A_B,
        A_C
    ],
    "Total population": [
        N_A,
        N_B,
        N_C
    ],
    "Reidentification rate (%)": [
        rate_A,
        rate_B,
        rate_C
    ]
})

q8_results["Reidentification rate (%)"] = q8_results[
    "Reidentification rate (%)"
].round(4)

q8_results

,Case,A - Unique identifiers,Total population,Reidentification rate (%)
0,Case A - All anonymized,4852,150000,3.2347
1,Case B - Keep power 0-1,43496,150000,28.9973
2,Case C - Keep power 0-2,149000,150000,99.3333


## Conclusion

The anonymization experiment demonstrates that generalization can
substantially reduce the reidentification rate.

When all predictor variables are generalized (Case A), the
reidentification rate is 3.2347%. When power-1 variables are kept
unchanged (Case B), the rate increases to 28.9973%. When all selected
predictors are kept unchanged (Case C), the rate reaches 99.3333%.

Therefore, the precision of the information retained in the dataset has a
strong impact on the uniqueness of individual records. Generalization
reduces the number of unique combinations while preserving a level of
information that can still be used for predictive modelling.

The anonymization strategy therefore represents a trade-off between
privacy protection and the preservation of useful information for the
prediction model.